# Pay for Data — Heurist Finance Agent

## Overview

In this use case we build a finance research agent that autonomously pays for real-time
market data using **Amazon Bedrock AgentCore payments**. The agent calls paid
[Heurist](https://heurist.xyz) endpoints for live prices, SEC filings, and macro
indicators, analyzes the data with AgentCore Code Interpreter, and exports charts and
reports — all without any manual payment code in the tools.

### Use Case Details

| Information | Details |
|:---|:---|
| Use case type | Agentic data retrieval with autonomous micropayments |
| Agent type | Single |
| Payment protocol | x402 (HTTP 402 Payment Required) |
| Agentic framework | [Strands Agents](https://strandsagents.com/) |
| LLM model | Claude Sonnet 4 on Amazon Bedrock (configurable) |
| Complexity | Intermediate |
| SDK used | `bedrock-agentcore[strands-agents]` (public PyPI) |
| Wallet type | Embedded crypto wallet (Coinbase CDP) |
| Payment network | Base mainnet (USDC) |

### Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│  Strands Agent (Claude on Bedrock)                                  │
│                                                                     │
│  ┌──────────────┐   ┌──────────────────────────────────────────┐    │
│  │ http_request │   │ AgentCorePaymentsPlugin                  │    │
│  │  (tool)      │   │                                          │    │
│  │              │──▶│  intercepts HTTP 402                     │    │
│  │  POST to     │   │  ↓                                       │    │
│  │  Heurist     │   │  PaymentManager.generate_payment_header  │    │ 
│  │  endpoint    │   │  ↓                                       │    │
│  │              │   │  AgentCore payments API                  │    │
│  │              │   │  (GetPaymentInstrument + ProcessPayment) │    │
│  │              │   │  ↓                                       │    │
│  │              │◀──│  retries with X-PAYMENT header           │    │
│  └──────────────┘   └──────────────────────────────────────────┘    │
│         │                                                           │
│         ▼                                                           │
│  ┌──────────────────────────┐                                       │
│  │ AgentCore Code           │                                       │
│  │ Interpreter              │                                       │
│  │ (pandas / matplotlib)    │                                       │
│  └──────────────────────────┘                                       │
│         │                                                           │
│         ▼                                                           │
│  heurist_finance_agent/artifacts/  (charts, reports)                │
└─────────────────────────────────────────────────────────────────────┘
                │
                ▼
  Heurist mesh (x402 endpoints)
  USDC settlement on Base mainnet
```

### AgentCore Payments Capabilities Demonstrated

This use case showcases the following AgentCore payments capabilities:

| Capability | How it is used here |
|:---|:---|
| **Payment manager** | Central resource that authorizes and tracks all payment activity. Its ARN is passed to `AgentCorePaymentsPluginConfig`. |
| **Payment instrument** | An embedded crypto wallet (Coinbase CDP, USDC on Base) registered as a credential. The agent draws from it per call. |
| **Payment session** | A time-bounded, budget-capped authorization (`maxSpendAmount`). The agent cannot spend beyond the session limit. |
| **Payment processing** | End-to-end x402 negotiation, proof generation, retry, and on-chain settlement — handled automatically by `AgentCorePaymentsPlugin`. |
| **Payment limits** | Developer-defined per-session spend cap enforced at runtime. Set via `maxSpendAmount` when creating the payment session. |

### Use Case Key Features

- HTTP 402 payment processing via `AgentCorePaymentsPlugin` — no manual payment code in tools
- Embedded wallet (Coinbase CDP) with USDC as the settlement asset
- AgentCore Code Interpreter for sandboxed pandas/matplotlib analysis and artifact export
- Payment limits enforced at the payment session scope
- Agent never holds private keys; signing is delegated to AgentCore payments

**Before running:**
1. `pip install -r requirements.txt`
2. `cp .env.example .env` and fill in your credentials (payment manager ARN, payment session ID, payment instrument ID)
3. Run the catalog sync cell below

See [`README.md`](README.md) for full setup details.

## Install dependencies

In [ ]:
%pip install -r requirements.txt --quiet

## Step 1 — Configure your environment

Load credentials from `.env`. Copy `.env.example` to `.env` and fill in the values
from the setup tutorial (`00-getting-started/`).

In [ ]:
from heurist_finance_agent.config import get_config

cfg = get_config()
print(f"✅ Region:             {cfg.aws_region}")
print(f"✅ Payment manager:    {cfg.payment_manager_arn}")
print(f"✅ Payment session:    {cfg.payment_session_id}")
print(f"✅ Payment instrument: {cfg.payment_instrument_id}")
print(f"✅ Model:              {cfg.bedrock_model_id}")

## Step 2 — Sync the Heurist tool catalog

Fetches the current registry of x402-enabled endpoints from the Heurist mesh and caches
it locally. The agent's system prompt is built from this catalog so it knows which URLs
to call, what parameters each tool accepts, and what each call costs.

In [ ]:
from heurist_finance_agent.catalog import fetch_live_catalog, get_tools_for_agents

catalog = fetch_live_catalog()
selected = get_tools_for_agents(cfg.heurist_tool_agent_ids)

print(f"Agents in registry: {catalog['count']}")
print(f"Selected agents:    {', '.join(cfg.heurist_tool_agent_ids)}")
print(f"Loaded paid tools:  {len(selected)}")
print()
for t in selected:
    print(f"  {t['agent_id']:30s}  {t['tool_name']:35s}  ${t['price_usd']:.3f}")

## Step 3 — Run the agent

The agent is configured in [`agent.py`](heurist_finance_agent/agent.py) with:

- **`http_request`** — calls Heurist x402 endpoints
- **`AgentCorePaymentsPlugin`** — intercepts HTTP 402 responses, asks the AgentCore
  payment manager to generate a payment proof against the configured payment instrument
  and payment session, attaches the proof as an `X-PAYMENT` header, and retries
- **AgentCore Code Interpreter** — sandboxed Python environment for pandas/matplotlib
  analysis and chart generation

The payment flow per tool call:
1. `http_request` sends a POST to the Heurist endpoint
2. Heurist returns HTTP 402 with x402 payment terms
3. `AgentCorePaymentsPlugin` intercepts the 402
4. The plugin asks the AgentCore payment manager to generate a payment proof
5. The payment manager uses the payment instrument to sign a USDC transfer and returns a proof
6. The plugin attaches the proof as `X-PAYMENT` and retries — Heurist returns the data

Change the prompt below to explore different queries.

In [ ]:
from heurist_finance_agent.agent import invoke_agent

# Default: macroeconomic summary (non-crypto, uses FredMacroAgent)
prompt = (
    "Use FredMacroAgent to fetch the latest US GDP growth rate and unemployment rate. "
    "Summarize the current macroeconomic environment in a brief markdown report."
)

result = invoke_agent(prompt)
print(result)

## Step 4 — Inspect artifacts

Charts and reports the agent produced are saved to `heurist_finance_agent/artifacts/`.

In [ ]:
from pathlib import Path

artifacts_dir = Path("heurist_finance_agent/artifacts")
files = sorted(p for p in artifacts_dir.glob("*") if p.is_file() and p.name != ".gitkeep")
if files:
    for path in files:
        print(f"{path.stat().st_size:>10} bytes  {path.name}")
else:
    print("No artifacts yet — run the agent cell above first.")

## Step 5 — Cleanup

The resources created for this use case are managed outside this notebook (payment manager,
payment instrument, and payment session are created in the setup tutorial). The only
local resources to clean up are the cached catalog and any generated artifacts.

### Local cleanup

In [ ]:
import shutil
from pathlib import Path

# Remove the cached Heurist catalog (will be re-fetched on next sync)
cache = Path("heurist_finance_agent/catalog_live_cache.json")
if cache.exists():
    cache.unlink()
    print(f"Removed {cache}")
else:
    print("No catalog cache to remove.")

# Optionally remove generated artifacts
artifacts_dir = Path("heurist_finance_agent/artifacts")
removed = []
for p in artifacts_dir.glob("*"):
    if p.is_file() and p.name != ".gitkeep":
        p.unlink()
        removed.append(p.name)
if removed:
    print(f"Removed artifacts: {', '.join(removed)}")
else:
    print("No artifacts to remove.")

### AWS resource cleanup

Payment sessions expire automatically when `expiryTimeInMinutes` elapses — no manual
deletion is required. To clean up the payment manager and payment instrument created
during setup, refer to the cleanup section of the setup tutorial
(`00-getting-started/00-setup-agentcore-payments/`).

---

## Shared Responsibility

This sample is provided for educational purposes. Before using AgentCore payments in
production, review the following responsibilities:

| Responsibility | AWS | You |
|:---|:---:|:---:|
| Securing the AgentCore payments service infrastructure | ✅ | |
| Encrypting payment credentials at rest (credential provider) | ✅ | |
| Enforcing payment session limits at the service level | ✅ | |
| Settling on-chain transactions (Coinbase CDP / Stripe Privy) | ✅ | |
| Configuring IAM roles with least-privilege permissions | | ✅ |
| Setting appropriate `maxSpendAmount` payment limits per session | | ✅ |
| Protecting `.env` files and AWS credentials from exposure | | ✅ |
| Funding the payment instrument with sufficient USDC | | ✅ |
| Monitoring agent spend and session usage | | ✅ |
| Validating prompts to prevent prompt injection attacks | | ✅ |
| Reviewing Heurist endpoint terms of service | | ✅ |

> **Security note:** Never commit `.env` files or private keys to source control.
> Use AWS Secrets Manager or environment-level injection for production credentials.
> Payment sessions are time-bounded and budget-capped — set conservative limits
> (`maxSpendAmount`) when running agents in automated or unattended contexts.